# AdaLQO: Adaptive Learned Query Optimizer with Shifting Detector

In [1]:
import argparse
import time
import os

import numpy as np
import pandas as pd
import torch
import sys
import AdaLQO.utils as utils
from AdaLQO.utils import plot_res
from AdaLQO.utils import prediction
from AdaLQO.shift_detector import mmd,ws,ks_values_pca

sys.path.insert(0, 'bao_server')
import bao_server.model as bao_model
import copy
import random

# sys.path.append('ShiftHandler')
# from replay_buffer import summarizer

In [2]:
df = pd.read_pickle("dataset/tpc-ds/data_df.pkl")
BATCH_SIZE = 100
data = utils.split_dataset(df, batch_size=BATCH_SIZE, random_state=None)

## Init Model


In [3]:
def train_bao_model(X,y):
    model = bao_model.BaoRegression(have_cache_data=False, verbose=False)
    model.fit_feature_extractor(X, y)
    model.fit_model(X, y, seed=42, ada_size=False)

    return model

def init_bao_model(train_data):
    X, y = utils.get_training_data(train_data)
    return X,y, train_bao_model(X, y)

def retrain_model(X, y, cur_data):
    X_cur, y_cur = utils.get_training_data(cur_data)

    new_X = X + X_cur
    new_y = y + y_cur

    return new_X, new_y, train_bao_model(new_X, new_y)

In [4]:
train_data = data[0][0]
# X,y,reg = init_bao_model(train_data)

## 3. Queries(Plans) Featurization

In [5]:
def embedding_plans(model, all_plans):
    trees = model._BaoRegression__tree_transform.transform(all_plans)
    return model._BaoRegression__net.get_fixed_features(trees)

def get_plans(queries):
    all_plans = []
    for plans in queries["plans"]:
        all_plans.extend(plans)
    return all_plans

## 4. Predict Other Group Data & Continual Learning

In [6]:
import bao_server.featurize as f

def safe_prediction(model, data):
    try:
        return prediction(model, data)
    except f.TreeBuilderError as e:
        print("TreeBuilderError during prediction:", e)
        print("Refitting feature extractor and model on current batch...")

        X = []
        y = []

        for _, row in data.iterrows():
            X.extend(row["plans"])
            y.extend(row["latency_list"])

        model.fit_feature_extractor(X, y)
        model.fit_model(X, y, seed=42, ada_size=False)

        return prediction(model, data)

### Relationship between MMD score and Prediction performance

In [7]:
def analyze_regret_detailed(dataset):
    batch_rows = []
    mmd_out_path="results/tpcds/regret_batch.csv"
    for i in range(len(dataset[0])):
        train_df = dataset[0][i]
        X_train, y_train, model = init_bao_model(train_df)
        base_embedding = embedding_plans(model, X_train)

        for j in range(len(dataset[0])):
            test_df = dataset[0][j]
            X_test = get_plans(test_df)
            cur_embedding = embedding_plans(model, X_test)
            cur = cur_embedding.cpu().detach().numpy()
            base = base_embedding.cpu().detach().numpy()
            with torch.no_grad():
                # mmd_score = mmd(cur_embedding, base_embedding)
                mmd_score = mmd(cur, base)
            ks_result = ks_values_pca(cur, base)
            ws_score = ws(cur, base)

            if hasattr(mmd_score, "detach"):
                mmd_value = float(mmd_score.detach().cpu().item())
            else:
                mmd_value = float(mmd_score)

            res = prediction(model, test_df)
            batch_rows.append({
                "train_batch": i,
                "test_batch": j,
                "mmd_score": mmd_value,
                "ks_result": ks_result,
                "ws_score": ws_score,
                "pred_res": res,
                "same_batch": i == j
            })

            print(
                f"train_batch={i}, test_batch={j}, "
                f"mmd={mmd_value:.4f}, "
                f"ks_stat={ks_result.statistic:.4f}, "
                f"ws_score={ws_score:.4f}, "
                f"mean_regret={res['regret'].mean():.4f}"
            )

    batch_df = pd.DataFrame(batch_rows)
    batch_df.to_csv(mmd_out_path, index=False)

    return batch_df

In [8]:
# IMPORTANT: To get mmd,ks,ws vs regret dataset
# analyze_regret_detailed(dataset=data)

In [9]:
def cl(dataset, shift_detect):
    res = []
    X, y, ori_reg = init_bao_model(dataset[0][0])
    base_embedding = embedding_plans(ori_reg, X)
    cur_reg = copy.deepcopy(ori_reg)
    retrain_counter = 0

    for i, phase in enumerate(data):
        for j, queries in enumerate(phase):
            if i+j == 0: continue
                # X,y, ori_reg = init_bao_model(queries)

            plans = get_plans(queries)
            cur_embedding = embedding_plans(cur_reg, plans)
            match shift_detect:
                case "mmd":
                    mmd_score = mmd(cur_embedding, base_embedding)
                    print(i, j, mmd_score)
                    # if detected shifting, retrain Bao Model with the last batch data
                    if mmd_score > 0.05:
                        X,y,cur_reg = retrain_model(X, y, queries)
                        base_embedding = embedding_plans(ori_reg, X)
                        retrain_counter += 1
                case "ks":
                    cur = cur_embedding.cpu().detach().numpy()
                    base = base_embedding.cpu().detach().numpy()
                    ks_score = ks_values_pca(cur, base)
                    print(i, j, ks_score)
                    # if detected shifting, retrain Bao Model with the last batch data
                    if ks_score.statistic > 0.2:
                        # retrain bao model
                        retrain_counter += 1
                case "ws":
                    cur = cur_embedding.cpu().detach().numpy()
                    base = base_embedding.cpu().detach().numpy()
                    ws_score = ws(cur, base)
                    print(i, j, ws_score)
                    # if detected shifting, retrain Bao Model with the last batch data
                    if ws_score > 0.5:
                        # retrain bao model
                        retrain_counter += 1
            try:
                ori_res = prediction(ori_reg, queries)
            except f.TreeBuilderError:
                ori_res = safe_prediction(ori_reg, queries)
            cur_res = safe_prediction(cur_reg, queries)
            cur_res["base_bao_latency"] = ori_res["bao_latency"]
            res.append(cur_res)
            pre_data = data[i][j]

            del mmd_score
            torch.cuda.empty_cache()

    return res


## 5. Maximum Mean Discrepancy (MMD)

In [10]:
res = cl(dataset=data, shift_detect="mmd")
res_df = pd.concat(res,ignore_index=True)
plot_res("shift detect", res_df, "results/tpcds/performance_10_mmd.png")

0 1 tensor(0.0190, device='cuda:0', grad_fn=<AddBackward0>)
0 2 tensor(0.0608, device='cuda:0', grad_fn=<AddBackward0>)
0 3 tensor(3.0177, device='cuda:0', grad_fn=<AddBackward0>)
0 4 tensor(3.7827, device='cuda:0', grad_fn=<AddBackward0>)
0 5 tensor(2.9132, device='cuda:0', grad_fn=<AddBackward0>)
0 6 tensor(3.1271, device='cuda:0', grad_fn=<AddBackward0>)
0 7 tensor(3.0995, device='cuda:0', grad_fn=<AddBackward0>)
0 8 tensor(3.2417, device='cuda:0', grad_fn=<AddBackward0>)
0 9 tensor(3.0349, device='cuda:0', grad_fn=<AddBackward0>)
0 10 tensor(2.9024, device='cuda:0', grad_fn=<AddBackward0>)
0 11 tensor(3.1909, device='cuda:0', grad_fn=<AddBackward0>)
0 12 tensor(3.4340, device='cuda:0', grad_fn=<AddBackward0>)
0 13 tensor(2.8273, device='cuda:0', grad_fn=<AddBackward0>)
0 14 tensor(2.9411, device='cuda:0', grad_fn=<AddBackward0>)
0 15 tensor(2.8360, device='cuda:0', grad_fn=<AddBackward0>)


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.56 GiB. GPU 0 has a total capacity of 4.00 GiB of which 0 bytes is free. Of the allocated memory 13.61 GiB is allocated by PyTorch, and 4.01 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

## 7. Kolmogorov-Smirnov (KS) test

In [ ]:
# reg = init_bao_model(data)
# res = cl(reg,"ks",data)
# res_df = pd.concat(res,ignore_index=True)
# plot_res("shift detect", res_df, "results/tpcds/performance_10_ks.png")

### KL-Divergence Or JSD vs Regrets (Optional)

## 9. Wassertein Distance vs Regrets

In [ ]:
# reg = init_bao_model(data)
# res = cl(reg,"ws",data)
# res_df = pd.concat(res,ignore_index=True)
# plot_res("shift detect", res_df, "results/tpcds/performance_20_ws.png")

Therefore, I recommend using the following main figures for the final paper:

Normalized Latency (Default/Bao/Optimal) ← Main Figure

Regret vs Phase ← Core Results

MMD Score + Retrain Point ← Method Validation

Top-1 Accuracy ← Auxiliary Results

These four figures should be sufficient to fully support the experimental section of the entire Continual LQO paper.
因此我建议最终论文主图用：
Normalized Latency (Default/Bao/Optimal) ← 主图
Regret vs Phase ← 核心结果
MMD Score + Retrain Point ← 方法验证
Top-1 Accuracy ← 辅助结果
这四张图基本就能完整支撑整个 Continual LQO 论文的实验部分。